# Silver — Extração de descrições únicas para classificação

Primeira etapa da Silver. **Não grava Silver ainda** — apenas lê o Bronze,
normaliza as descrições e exporta um Excel com todas as descrições únicas para
classificação manual nos 8 grupos econômicos.

Fluxo do projeto:
1. **(este notebook)** normaliza → extrai descrições únicas → Excel
2. classificar o Excel nos 8 grupos (vira `silver.dim_classificacao`)
3. ingerir a dimensão + montar a Silver completa com o join do `grupo_economico`

**Normalização da `descricao_canonica`:** minúsculas, sem acento, espaços
colapsados, hífen preservado. Remove o número do item (`3.1 - `) e tabs.

## 1. Imports e leitura do Bronze

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from db import get_engine

engine = get_engine()

# Lê a camada Bronze inteira
df = pd.read_sql("SELECT * FROM bronze.custo_soja_raw", engine)
print(f"Bronze: {len(df):,} linhas")
df.head(3)

Bronze: 23,727 linhas


,descricao_custo,categoria,(R$/ha),(R$/60 kg),produtividade_media(kg/ha),local,estado,ano,tipo_cultivo,aba_origem,linha_origem
0,1 - Operação com aviões,I - DESPESAS DE CUSTEIO DA LAVOURA,0.00,0.00,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,11
1,2 - Operação com máquinas,I - DESPESAS DE CUSTEIO DA LAVOURA,143.41,2.99,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,12
2,3 - Mão-de-obra temporária,I - DESPESAS DE CUSTEIO DA LAVOURA,5.85,0.12,Produtividade media: |2880.0,Barreiras,BA,2006,convencional,Barreiras-BA-2006,13


## 2. Normalização das descrições

Separa o número do item da descrição e gera a forma canônica. Mantém o hífen
(então `mao de obra` e `mao-de-obra` continuam distintos — a decisão de tratá-los
como o mesmo grupo é feita na classificação manual, não aqui).

In [2]:
RE_NUM_PREFIX = re.compile(r"^\s*\d+(\.\d+)?\s*-\s*")


def normaliza(desc_crua: str):
    """Retorna (descricao_limpa, descricao_canonica) a partir da descrição crua."""
    s = str(desc_crua).replace("\t", "")
    # remove o prefixo numérico do item (3.1 - , 10 - ...)
    m = RE_NUM_PREFIX.match(s)
    if m:
        s = s[m.end():]
    desc_limpa = s.strip().rstrip(":").strip()
    # canônica: minúsculas + sem acento + espaços colapsados (hífen mantido)
    canon = unicodedata.normalize("NFKD", desc_limpa.lower())
    canon = "".join(ch for ch in canon if not unicodedata.combining(ch))
    canon = re.sub(r"\s+", " ", canon).strip()
    return desc_limpa, canon


df[["descricao_limpa", "descricao_canonica"]] = df["descricao_custo"].apply(
    lambda x: pd.Series(normaliza(x))
)

# conferência
df[["descricao_custo", "descricao_limpa", "descricao_canonica"]].head(10)

,descricao_custo,descricao_limpa,descricao_canonica
0,1 - Operação com aviões,Operação com aviões,operacao com avioes
1,2 - Operação com máquinas,Operação com máquinas,operacao com maquinas
2,3 - Mão-de-obra temporária,Mão-de-obra temporária,mao-de-obra temporaria
3,4 - Mão-de-obra fixa,Mão-de-obra fixa,mao-de-obra fixa
4,5 - Sementes,Sementes,sementes
5,6 - Fertilizantes,Fertilizantes,fertilizantes
6,7 - Defensivos,Defensivos,defensivos
7,TOTAL DAS DESPESAS DE CUSTEIO DA LAVOURA (A),TOTAL DAS DESPESAS DE CUSTEIO DA LAVOURA (A),total das despesas de custeio da lavoura (a)
8,1 - Transporte externo,Transporte externo,transporte externo
9,"2 - Recepção, limpeza, secagem, armazenagem ...","Recepção, limpeza, secagem, armazenagem 30-d","recepcao, limpeza, secagem, armazenagem 30-d"


## 3. Agregar descrições únicas

Uma linha por `descricao_canonica`, com frequência e um exemplo da forma original crua (apoio à classificação).

In [3]:
agg = (
    df.groupby("descricao_canonica")
      .agg(
          frequencia=("descricao_canonica", "size"),
          descricao_exemplo=("descricao_limpa", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
          exemplo_original_cru=("descricao_custo", lambda s: s.iloc[0]),
      )
      .reset_index()
      .sort_values("frequencia", ascending=False)
      .reset_index(drop=True)
)

# coluna vazia para a classificação manual
agg.insert(1, "grupo_economico", "")
# torna visível o \t no exemplo cru
agg["exemplo_original_cru"] = agg["exemplo_original_cru"].str.replace("\t", "\\t", regex=False)

print(f"Descrições únicas a classificar: {len(agg)}")
agg.head(20)

Descrições únicas a classificar: 115


,descricao_canonica,grupo_economico,frequencia,descricao_exemplo,exemplo_original_cru
0,depreciacao de implementos,,515,Depreciação de implementos,2 - Depreciação de implementos
1,depreciacao de benfeitorias/instalacoes,,515,Depreciação de benfeitorias/instalações,1 - Depreciação de benfeitorias/instalações
2,depreciacao de maquinas,,515,Depreciação de Máquinas,3 - Depreciação de máquinas
3,encargos sociais,,515,Encargos Sociais,2 - Encargos sociais
4,total de outros custos fixos (f),,515,TOTAL DE OUTROS CUSTOS FIXOS (F),Total de Outros Custos Fixos (F)
5,total de depreciacoes (e),,515,TOTAL DE DEPRECIAÇÕES (E),Total de Depreciações (E)
6,total das despesas financeiras (c),,515,TOTAL DAS DESPESAS FINANCEIRAS (C),Total das Despesas Financeiras (C)
7,seguro do capital fixo,,515,Seguro do capital fixo,3 - Seguro do capital fixo
8,assistencia tecnica,,507,Assistência Técnica,4 - Assistência Técnica
9,transporte externo,,507,Transporte Externo,1 - Transporte externo


## 4. Exportar Excel para classificação

Gera `descricoes_para_classificar.xlsx` com um **dropdown** dos 8 grupos na coluna
`grupo_economico`, para você classificar sem digitar (evita erro de grafia).

In [ ]:
#%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

GRUPOS = [
    "Sementes", "Fertilizantes", "Defensivos", "Mecanizacao", "Mao de obra",
    "Pos-colheita", "Custos financeiros e seguros", "Terra e outros", "Nao classificado",
]

SAIDA = Path("..") / "dados" / "dim" / "descricoes_para_classificar.xlsx"
SAIDA.parent.mkdir(parents=True, exist_ok=True)

wb = Workbook()
ws = wb.active
ws.title = "classificacao"

cols = list(agg.columns)
ws.append(cols)
for c in range(1, len(cols) + 1):
    cell = ws.cell(1, c)
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="2F5496")
    cell.alignment = Alignment(horizontal="center")

for _, r in agg.iterrows():
    ws.append([r[c] for c in cols])

# dropdown dos 8 grupos na coluna B (grupo_economico)
dv = DataValidation(type="list", formula1='"' + ",".join(GRUPOS) + '"', allow_blank=True)
ws.add_data_validation(dv)
dv.add(f"B2:B{len(agg) + 1}")

for col, w in {"A": 38, "B": 30, "C": 12, "D": 34, "E": 40}.items():
    ws.column_dimensions[col].width = w
ws.freeze_panes = "A2"

wb.save(SAIDA)
print(f"Excel gerado: {SAIDA.resolve()}")
print(f"{len(agg)} descrições — classifique a coluna grupo_economico e me envie de volta.")

Excel gerado: C:\Users\cauai.Capozzoli\Desktop\projetos\repo-local\custos_producao\dados\dim\descricoes_para_classificar.xlsx
115 descrições — classifique a coluna grupo_economico e me envie de volta.


---
# Parte 2 — Ingestão da dimensão de classificação

Depois de classificar o Excel exportado acima, salve-o como
**`descricoes_para_classificar_preenchido.xlsx`** (arquivo distinto do de saída,
para nunca sobrescrever) em `dados/dim/` e rode as células abaixo.

Esta parte lê o Excel preenchido, valida a qualidade da classificação e grava
`silver.dim_classificacao` no Postgres. **Idempotente:** recria a tabela do Excel
a cada execução — corrigir a classificação é editar o Excel e rerodar.

A dimensão guarda 4 colunas: a chave de join, o grupo, e duas de apoio para
auditoria (`frequencia`, `descricao_exemplo`).

## 2.1 Ler o Excel preenchido

In [16]:
from sqlalchemy import text

In [17]:
ENTRADA = Path("..") / "dados" / "dim" / "descricoes_para_classificar_preenchido.xlsx"
assert ENTRADA.exists(), (
    f"Não encontrei {ENTRADA.resolve()}.\n"
    "Classifique o Excel exportado na Parte 1 e salve-o com este nome antes de rodar."
)

dim = pd.read_excel(ENTRADA, engine="openpyxl")
print(f"Lidas {len(dim)} descrições classificadas")
dim.head(5)

Lidas 115 descrições classificadas


,descricao_canonica,grupo_economico,frequencia,descricao_exemplo,exemplo_original_cru
0,depreciacao de implementos,Custos financeiros e seguros,515,Depreciação de implementos,2 - Depreciação de implementos
1,depreciacao de benfeitorias/instalacoes,Custos financeiros e seguros,515,Depreciação de benfeitorias/instalações,1 - Depreciação de benfeitorias/instalações
2,depreciacao de maquinas,Custos financeiros e seguros,515,Depreciação de Máquinas,3 - Depreciação de máquinas
3,encargos sociais,Mao de obra,515,Encargos Sociais,2 - Encargos sociais
4,total de outros custos fixos (f),Nao classificado,515,TOTAL DE OUTROS CUSTOS FIXOS (F),Total de Outros Custos Fixos (F)


In [18]:
GRUPOS_VALIDOS = {
    "Sementes", "Fertilizantes", "Defensivos", "Mecanizacao", "Mao de obra",
    "Pos-colheita", "Custos financeiros e seguros", "Terra e outros", "Nao classificado",
}

problemas = []

# 1. grupos vazios
vazios = dim["grupo_economico"].isna().sum()
if vazios:
    problemas.append(f"{vazios} descrição(ões) sem grupo preenchido")

# 2. grupos fora do domínio
invalidos = dim[~dim["grupo_economico"].isin(GRUPOS_VALIDOS)]
if len(invalidos):
    problemas.append(f"{len(invalidos)} grupo(s) inválido(s): {invalidos['grupo_economico'].unique().tolist()}")

# 3. chaves duplicadas
dups = dim["descricao_canonica"].duplicated().sum()
if dups:
    problemas.append(f"{dups} descricao_canonica duplicada(s)")

# 4. cobertura contra o Bronze (usa o df já normalizado da Parte 1)
canon_bronze = set(df["descricao_canonica"].unique())
canon_dim = set(dim["descricao_canonica"].unique())
orfaos = canon_bronze - canon_dim
if orfaos:
    problemas.append(f"{len(orfaos)} descrição(ões) do Bronze SEM classificação: {sorted(orfaos)[:5]}...")

if problemas:
    print("ATENÇÃO — problemas encontrados:")
    for p in problemas:
        print("  -", p)
    raise ValueError("Corrija o Excel antes de gravar a dimensão.")
else:
    print("Validação OK — dimensão íntegra e cobre 100% do Bronze.")
    print("\nDistribuição por grupo:")
    print(dim["grupo_economico"].value_counts().to_string())

Validação OK — dimensão íntegra e cobre 100% do Bronze.

Distribuição por grupo:
grupo_economico
Custos financeiros e seguros    25
Nao classificado                20
Mao de obra                     18
Mecanizacao                     13
Terra e outros                  12
Pos-colheita                    11
Sementes                        10
Fertilizantes                    4
Defensivos                       2


## 2.3 Gravar `silver.dim_classificacao`

Idempotente: recria a tabela a cada execução. Mantém só as 4 colunas da dimensão.

In [19]:
dim_final = dim[["descricao_canonica", "grupo_economico", "frequencia", "descricao_exemplo"]].copy()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS silver.dim_classificacao;"))

dim_final.to_sql(
    name="dim_classificacao",
    schema="silver",
    con=engine,
    if_exists="replace",
    index=False,
)
print(f"Gravado silver.dim_classificacao: {len(dim_final)} linhas")

Gravado silver.dim_classificacao: 115 linhas
